In [50]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
import matplotlib.pyplot as plt
from package.RankAMIP.logistic import run_logistic_regression
from package.RankAMIP.data_script import make_BT_design_matrix
from package.RankAMIP.logistic import LogisticAMIP
from package.RankAMIP.logistic import find_closest_matchups
from package.RankAMIP.logistic import isRankingRobust
from package.RankAMIP.data_script import *

### Load Data

In [51]:
# Import datasets from
df = pd.concat([pd.read_csv(f'https://raw.githubusercontent.com/JeffSackmann/tennis_atp/refs/heads/master/atp_matches_{i}.csv') for i in [2020, 2021, 2022, 2023]], ignore_index=True)

#df = pd.read_csv('https://raw.githubusercontent.com/JeffSackmann/tennis_atp/refs/heads/master/atp_matches_2020.csv')

In [ ]:
df_2024_rankings = pd.read_csv('https://raw.githubusercontent.com/JeffSackmann/tennis_atp/refs/heads/master/atp_rankings_current.csv')

In [ ]:
# inspect the available splits
df.head()

In [ ]:
top_10_ranked_playerids = df_2024_rankings['player'].head(30)
top_10_ranked_playerids

Filter for games between top-ranked players, where both the winner_id and loser_id are in top_10_ranked_playerids.

In [ ]:
top_matchups = df[df['winner_id'].isin(top_10_ranked_playerids) & df['loser_id'].isin(top_10_ranked_playerids)]
top_matchups.shape

Filter any singletons

In [ ]:
game_counts = top_matchups['winner_id'].value_counts() + top_matchups['loser_id'].value_counts()
valid_players = game_counts[game_counts > 30].index

# Filter rows where both winner and loser are in valid_players
top_matchups = top_matchups[top_matchups['winner_id'].isin(valid_players) & top_matchups['loser_id'].isin(valid_players)]

In [ ]:
top_matchups.shape

We will choose to assign player_A and player_B based on alphabetical ordering of name.

In [ ]:
rawBT = top_matchups[['winner_name', 'loser_name', 'winner_rank', 'loser_rank']]
rawBT['player_A'] = rawBT.apply(lambda x: min(x['winner_name'], x['loser_name']), axis=1)
rawBT['player_B'] = rawBT.apply(lambda x: max(x['winner_name'], x['loser_name']), axis=1)
rawBT.head()

In [ ]:
rawBT['winner_player_A'] = (rawBT['player_A'] == rawBT['winner_name']).astype(int)
rawBT.head()

In [ ]:
# Count wins for each player
win_counts = rawBT['winner_name'].value_counts()
win_counts

Create winner_player_a column.

In [ ]:
rawBT.head()

In [ ]:
rawBT['winner_player_a'] = rawBT.apply(lambda x: 1 if x['winner_name'] == x['player_A'] else 0, axis=1)

In [ ]:
for_BT = rawBT[['player_A', 'player_B', 'winner_player_a']]
for_BT.head()
# rawBT_noTies.head() # (2575, 3)
# rawBT_noTies.shape

In [ ]:
# make the BT design matrix.
X, y, player_to_id = make_BT_design_matrix(for_BT, weight_tie = False)
X.shape, y.shape

In [ ]:
id_to_player = {v: k for k, v in player_to_id.items()}
id_to_player

In [ ]:
# compute BT scores.
model_full = run_logistic_regression(X, y)

# prepend model 0, the reference model, which has score 0.
bt_scores = np.insert(model_full.coef_[0], 0, 0)

In [ ]:
bt_scores

In [ ]:
# combine bt_scores with player names
bt_scores_with_names = {id_to_player[i]: score for i, score in enumerate(bt_scores)}
dict(sorted(bt_scores_with_names.items(), key=lambda x: x[1], reverse=True))


If we run a plain BT-model on the full data, Djokovic does not rank in the top spot.
When we count the wins on the restricted arena (restricted to games between top 10 players) Djokovic also does not come out with the most wins.

In [ ]:
# Count wins for each player
win_counts = rawBT['winner_name'].value_counts()
win_counts

#### Run Top-k Robustness Check.

In [ ]:
ks = [1, 5]
results = {}
for k in ks:
    alphaN = 1
    chatbotA = -1
    while chatbotA == -1:
        chatbotA, chatbotB, chatbotOriginalBetaDiff, chatNewBetaDiff, chatIndices = isRankingRobust(k, alphaN, X, y, weighted = False)
        results[(k, alphaN)] = (chatbotA, chatbotB, chatbotOriginalBetaDiff, chatNewBetaDiff, chatIndices)
        alphaN += 1

In [ ]:
# find the (k, alpha N) pairs that are non-robust.
results_nonrobust = {k: v for k, v in results.items() if v[0] != -1}
results_nonrobust

Restricting the arena to the top 10 players, we only need to drop 3 out of 67 matches.

In [ ]:
from package.RankAMIP.plot_util import *
rankings = return_rankings_list(X, y, results, 1, 1, player_to_id)

In [ ]:
# plot the rankings on the original arena
filename_to_save = 'fig/tennis3.png'
plot_title = 'Player Rankings in Tennis'
plot_bt_scores(X, y, rankings, alphaN, 10, plot_title, filename_to_save)